In [34]:
import pandas as pd 
import numpy as np 
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [35]:
df = pd.read_csv("Cleaned_data_save.csv")

In [36]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'A_EDA.ipynb.ipynb', 'B_Model_Building.ipynb', 'Churn_Modelling.csv', 'Cleaned_data_save.csv', 'C_Prediction.ipynb', 'model.pkl']


In [37]:
X = df.drop(columns = 'Exited')
X.head()

Y = df['Exited']
Y.head()

0    1
1    0
2    1
3    0
4    0
Name: Exited, dtype: int64

In [38]:
from sklearn.model_selection import train_test_split

In [40]:
X_train,X_test,Y_train,Y_test = train_test_split(
X,Y,test_size=0.2,random_state=42)

#### 2. Pipelines
Numerical Pipeline

### Missing Value

↓

Scaling

In [41]:
X = df.drop(["Exited", "RowNumber", "CustomerId", "Surname"], axis=1)
y = df["Exited"]

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

#### Step 2: Numerical aur Categorical Columns

In [43]:
num_col = X.select_dtypes(include=["int64", "float64"]).columns

cat_col = X.select_dtypes(include=["object"]).columns

print(num_col)
print(cat_col)

Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary'],
      dtype='object')
Index(['Geography', 'Gender'], dtype='object')


##### Step 3: Numerical Pipeline

In [44]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [45]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [46]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_col),
        ("cat", cat_pipeline, cat_col)
    ]
)

In [47]:
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['Geography', 'Gender'], dtype='object'))])

### Logistic Regression Pipelin

In [48]:
LR = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(random_state=42))
])

LR.fit(X_train, Y_train)
Y_pred_lr = LR.predict(X_test)

print("Logistic Accuracy:", accuracy_score(Y_test, Y_pred_lr))

Logistic Accuracy: 0.811


#### Prediction

In [49]:
y_pred_lr = LR.predict(X_test)

In [50]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(Y_test, Y_pred_lr))

Accuracy: 0.811


In [51]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, Y_pred_lr))

              precision    recall  f1-score   support

           0       0.83      0.96      0.89      1607
           1       0.55      0.20      0.29       393

    accuracy                           0.81      2000
   macro avg       0.69      0.58      0.59      2000
weighted avg       0.78      0.81      0.77      2000



In [52]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(Y_test, Y_pred_lr))

[[1543   64]
 [ 314   79]]


In [53]:
DT = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

In [54]:
DT.fit(X_train, Y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Geography', 'Gender'], dtype='object'))])),
                ('model', DecisionTreeClassifier(random_state=42))])

In [55]:
Y_pred_dt = DT.predict(X_test)

In [56]:
from sklearn.metrics import accuracy_score

print("Decision Tree Accuracy:", accuracy_score(Y_test, Y_pred_dt))

Decision Tree Accuracy: 0.7805


In [57]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, Y_pred_dt))

              precision    recall  f1-score   support

           0       0.88      0.85      0.86      1607
           1       0.45      0.51      0.48       393

    accuracy                           0.78      2000
   macro avg       0.66      0.68      0.67      2000
weighted avg       0.79      0.78      0.79      2000



In [58]:
from sklearn.neighbors import KNeighborsClassifier

KNN = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier())
])

KNN.fit(X_train, Y_train)

Y_pred_knn = KNN.predict(X_test)

print("KNN Accuracy:", accuracy_score(Y_test, Y_pred_knn))
print(classification_report(Y_test, Y_pred_knn))
print(confusion_matrix(Y_test, Y_pred_knn))

KNN Accuracy: 0.8365
              precision    recall  f1-score   support

           0       0.87      0.94      0.90      1607
           1       0.63      0.40      0.49       393

    accuracy                           0.84      2000
   macro avg       0.75      0.67      0.70      2000
weighted avg       0.82      0.84      0.82      2000

[[1515   92]
 [ 235  158]]


# Random Forest

In [59]:
from sklearn.ensemble import RandomForestClassifier

RF = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

RF.fit(X_train, Y_train)

Y_pred_rf = RF.predict(X_test)

print("Random Forest Accuracy :", accuracy_score(Y_test, Y_pred_rf))
print(classification_report(Y_test, Y_pred_rf))
print(confusion_matrix(Y_test, Y_pred_rf))

Random Forest Accuracy : 0.8635
              precision    recall  f1-score   support

           0       0.88      0.96      0.92      1607
           1       0.75      0.46      0.57       393

    accuracy                           0.86      2000
   macro avg       0.81      0.71      0.75      2000
weighted avg       0.85      0.86      0.85      2000

[[1545   62]
 [ 211  182]]


# Naive Bayes

In [61]:
from sklearn.naive_bayes import GaussianNB

X_train_nb = preprocessor.fit_transform(X_train)
X_test_nb = preprocessor.transform(X_test)

# Sparse ko Dense me convert karo
X_train_nb = X_train_nb.toarray()
X_test_nb = X_test_nb.toarray()

NB = GaussianNB()

NB.fit(X_train_nb, Y_train)

Y_pred_nb = NB.predict(X_test_nb)

print("Naive Bayes Accuracy :", accuracy_score(Y_test, Y_pred_nb))
print(classification_report(Y_test, Y_pred_nb))
print(confusion_matrix(Y_test, Y_pred_nb))

AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

#### SVM

In [63]:
from sklearn.svm import SVC

SVM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVC())
])

SVM.fit(X_train, Y_train)

Y_pred_svm = SVM.predict(X_test)

print("SVM Accuracy :", accuracy_score(Y_test, Y_pred_svm))
print(classification_report(Y_test, Y_pred_svm))
print(confusion_matrix(Y_test, Y_pred_svm))

SVM Accuracy : 0.8575
              precision    recall  f1-score   support

           0       0.86      0.98      0.92      1607
           1       0.79      0.37      0.51       393

    accuracy                           0.86      2000
   macro avg       0.83      0.67      0.71      2000
weighted avg       0.85      0.86      0.84      2000

[[1569   38]
 [ 247  146]]


# Model Comparison

In [64]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "KNN",
        "Random Forest",
        "Naive Bayes",
        "SVM"
    ],
    "Accuracy": [
        accuracy_score(Y_test, Y_pred_lr),
        accuracy_score(Y_test, Y_pred_dt),
        accuracy_score(Y_test, Y_pred_knn),
        accuracy_score(Y_test, Y_pred_rf),
        accuracy_score(Y_test, Y_pred_nb),
        accuracy_score(Y_test, Y_pred_svm)
    ]
})

results.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy
3,Random Forest,0.8635
5,SVM,0.8575
2,KNN,0.8365
0,Logistic Regression,0.8110
1,Decision Tree,0.7805
4,Naive Bayes,0.3530


# Best Model Save

In [65]:
import pickle

pickle.dump(SVM, open("model.pkl", "wb"))

In [66]:
import pickle

pickle.dump(SVM, open("model.pkl", "wb"))
print("Model Saved Successfully")

Model Saved Successfully
